# Capítulo 8 — Exercício 9: MNIST com e sem PCA

Vamos comparar **RandomForestClassifier** e **SGDClassifier** usando os pixels originais e componentes principais que preservam pelo menos **95% da variância do treino**.

O exercício usa as primeiras 60.000 imagens para treino e as 10.000 restantes para teste. Mediremos acurácia e tempo de treinamento, além do custo do PCA. Execute as células em ordem.

Referência: [notebook do autor, exercício 9](https://github.com/ageron/handson-ml3/blob/main/08_dimensionality_reduction.ipynb#9.).

## 1. Preparar o ambiente

Fixamos a semente e limitamos o paralelismo a quatro threads/processos de trabalho. Os tempos dependem do computador e da carga durante a execução; a semente ajuda a reproduzir os resultados, mas não garante tempos idênticos.

Usaremos os pixels sem padronização adicional, como no experimento do livro. Assim, a comparação mede o efeito de inserir PCA nessa configuração. Os mesmos hiperparâmetros serão usados antes e depois da redução, sem ajuste pelo conjunto de teste.

In [1]:
import os

N_JOBS = 4
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[variable] = str(N_JOBS)

import platform
from datetime import datetime
from time import perf_counter
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import sklearn
from IPython.display import Markdown, display
from sklearn.datasets import fetch_openml
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from threadpoolctl import threadpool_limits

thread_limits = threadpool_limits(limits=N_JOBS)
RANDOM_STATE = 42
executado_em = datetime.now(ZoneInfo("America/Recife")).isoformat(timespec="seconds")
print(f"Execução: {executado_em}")
print(f"Python {platform.python_version()} | NumPy {np.__version__} | scikit-learn {sklearn.__version__}")
print(f"Paralelismo configurado: {N_JOBS}")

Execução: 2026-10-19T16:52:48-03:00
Python 3.11.14 | NumPy 2.1.3 | scikit-learn 1.5.2
Paralelismo configurado: 4


## 2. Carregar e separar o MNIST

O OpenML fornece 70.000 imagens com 784 pixels cada (28 × 28). A primeira execução pode exigir download. Convertemos os pixels para `float32` para reduzir o uso de memória e os rótulos para inteiros.

Mantemos a ordem original e fazemos a divisão exigida pelo enunciado, sem embaralhar nem misturar os conjuntos.

In [2]:
inicio = perf_counter()
X, y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False, parser="auto")
X = X.astype(np.float32)
y = y.astype(np.uint8)
X_train, X_test = X[:60_000], X[60_000:]
y_train, y_test = y[:60_000], y[60_000:]

assert X_train.shape == (60_000, 784)
assert X_test.shape == (10_000, 784)
assert set(np.unique(y_train)) == set(range(10))
print(f"Treino: {X_train.shape} | Teste: {X_test.shape}")
print(f"Carregamento e preparação: {perf_counter() - inicio:.2f} s")

Treino: (60000, 784) | Teste: (10000, 784)
Carregamento e preparação: 4.56 s


## 3. Definir uma medição comum

Medimos somente `fit()` na coluna de treinamento. A predição no teste é cronometrada separadamente. A acurácia é a fração de imagens classificadas corretamente. Não usamos o teste para escolher parâmetros ou o número de componentes.

In [3]:
resultados = []

def avaliar(nome, representacao, modelo, treino, teste):
    inicio = perf_counter()
    modelo.fit(treino, y_train)
    tempo_treino = perf_counter() - inicio

    inicio = perf_counter()
    previsoes = modelo.predict(teste)
    tempo_predicao = perf_counter() - inicio
    resultado = {
        "Modelo": nome,
        "Dados": representacao,
        "Dimensões": treino.shape[1],
        "Treino (s)": tempo_treino,
        "Predição (s)": tempo_predicao,
        "Acurácia": accuracy_score(y_test, previsoes),
    }
    resultados.append(resultado)
    print(f"{nome} / {representacao}: treino={tempo_treino:.2f} s | acurácia={resultado['Acurácia']:.4%}")
    if hasattr(modelo, "n_iter_"):
        print(f"Épocas do SGD: {modelo.n_iter_}")
    return modelo

rf_params = dict(n_estimators=100, random_state=RANDOM_STATE, n_jobs=N_JOBS)
sgd_params = dict(loss="hinge", penalty="l2", alpha=0.0001,
                  max_iter=1000, tol=1e-3, random_state=RANDOM_STATE)

## 4. Treinar a floresta aleatória com os 784 pixels

Este é o resultado de referência da floresta. Cada árvore usa divisões alinhadas aos atributos; mudar a representação com PCA também muda as divisões que ficam disponíveis ao modelo.

In [4]:
rf_original = avaliar("Floresta aleatória", "Originais", RandomForestClassifier(**rf_params), X_train, X_test)

Floresta aleatória / Originais: treino=6.38 s | acurácia=97.0500%


## 5. Ajustar PCA preservando 95% da variância

`n_components=0.95` escolhe automaticamente o menor número de componentes necessário para atingir o limiar. Usamos a decomposição completa (`svd_solver="full"`) e `whiten=False`.

O PCA é ajustado **somente no treino**. Aplicamos a mesma transformação ao teste. Cronometramos o ajuste, a transformação do treino e a transformação do teste separadamente.

In [5]:
pca = PCA(n_components=0.95, svd_solver="full", whiten=False)
inicio = perf_counter()
pca.fit(X_train)
tempo_ajuste_pca = perf_counter() - inicio

inicio = perf_counter()
X_train_pca = pca.transform(X_train)
tempo_transformacao_treino = perf_counter() - inicio

inicio = perf_counter()
X_test_pca = pca.transform(X_test)
tempo_transformacao_teste = perf_counter() - inicio

variancia_preservada = float(pca.explained_variance_ratio_.sum())
assert variancia_preservada >= 0.95
assert pca.explained_variance_ratio_[:-1].sum() < 0.95
assert X_train_pca.shape[1] == X_test_pca.shape[1] == pca.n_components_
print(f"Dimensões: 784 → {pca.n_components_}")
print(f"Variância preservada no treino: {variancia_preservada:.4%}")
print(f"Ajuste PCA: {tempo_ajuste_pca:.2f} s")
print(f"Transformação do treino: {tempo_transformacao_treino:.2f} s")
print(f"Transformação do teste: {tempo_transformacao_teste:.2f} s")

Dimensões: 784 → 154
Variância preservada no treino: 95.0196%
Ajuste PCA: 1.13 s
Transformação do treino: 0.11 s
Transformação do teste: 0.01 s


## 6. Treinar a floresta com os componentes principais

Mantemos os hiperparâmetros da floresta. Reduzir o número de atributos não garante treinamento mais rápido: os novos atributos são combinações dos pixels, e a geometria pode ser menos favorável às divisões das árvores.

In [6]:
rf_pca = avaliar("Floresta aleatória", "PCA 95%", RandomForestClassifier(**rf_params), X_train_pca, X_test_pca)

Floresta aleatória / PCA 95%: treino=19.11 s | acurácia=94.8800%


## 7. Repetir com SGDClassifier

O SGD ajusta um classificador linear por gradiente descendente estocástico. Usamos a perda `hinge` e os mesmos parâmetros nas duas representações. O custo por época depende do número de atributos, mas a representação também pode alterar a convergência.

Esta comparação vale para a configuração escolhida. Padronizar atributos e ajustar hiperparâmetros em validação constituiriam outro experimento.

In [7]:
sgd_original = avaliar("SGDClassifier", "Originais", SGDClassifier(**sgd_params), X_train, X_test)

SGDClassifier / Originais: treino=64.97 s | acurácia=88.4700%
Épocas do SGD: 315


In [8]:
sgd_pca = avaliar("SGDClassifier", "PCA 95%", SGDClassifier(**sgd_params), X_train_pca, X_test_pca)

SGDClassifier / PCA 95%: treino=13.95 s | acurácia=88.1300%
Épocas do SGD: 175


## 8. Comparar os resultados

A tabela separa o treinamento do classificador do custo de preparação com PCA. A coluna de custo total para treino inclui ajuste do PCA, transformação do treino e ajuste do classificador. Não inclui download, carregamento ou avaliação no teste.

O PCA é compartilhado entre os dois modelos nesta execução. Para comparar cada alternativa de forma independente, contabilizamos seu custo integral em cada linha com PCA.

In [9]:
tabela = pd.DataFrame(resultados)
custo_pca_treino = tempo_ajuste_pca + tempo_transformacao_treino
tabela["PCA para treino (s)"] = np.where(tabela["Dados"] == "PCA 95%", custo_pca_treino, 0.0)
tabela["Total para treino (s)"] = tabela["Treino (s)"] + tabela["PCA para treino (s)"]
display(tabela.round(4))

,Modelo,Dados,Dimensões,Treino (s),Predição (s),Acurácia,PCA para treino (s),Total para treino (s)
0,Floresta aleatória,Originais,784,6.3789,0.0559,0.9705,0.0000,6.3789
1,Floresta aleatória,PCA 95%,154,19.1062,0.0551,0.9488,1.2385,20.3448
2,SGDClassifier,Originais,784,64.9687,0.0050,0.8847,0.0000,64.9687
3,SGDClassifier,PCA 95%,154,13.9509,0.0013,0.8813,1.2385,15.1894


## 9. Interpretar a execução

Abaixo, geramos a resposta com os números desta execução. O fator de aceleração é tempo original dividido pelo tempo com PCA: acima de 1 significa aceleração; abaixo de 1 significa desaceleração. A diferença de acurácia é expressa em pontos percentuais (p.p.).

Tempo e acurácia são medições de uma única execução, não médias de múltiplas repetições. A variância preservada é uma propriedade do treino e não uma garantia de preservar a acurácia.

In [10]:
linhas = [
    f"Execução em {executado_em}, com Python {platform.python_version()}, NumPy {np.__version__} e scikit-learn {sklearn.__version__}; paralelismo limitado a {N_JOBS}.",
    "",
    "Foram usadas as primeiras 60.000 imagens do MNIST para treino e as 10.000 restantes para teste. O PCA foi ajustado apenas no treino, com `svd_solver=\"full\"` e `whiten=False`. Os pixels foram convertidos para `float32`, sem padronização adicional.",
    "",
    f"O PCA reduziu **784 para {pca.n_components_} dimensões**, preservando **{variancia_preservada:.4%}** da variância do treino. O ajuste levou **{tempo_ajuste_pca:.2f} s**, a transformação do treino **{tempo_transformacao_treino:.2f} s** e a do teste **{tempo_transformacao_teste:.2f} s**.",
    "",
    "| Modelo | Dados | Dimensões | Treino (s) | Acurácia no teste | Total para treino (s) |",
    "| --- | --- | ---: | ---: | ---: | ---: |",
]
for _, r in tabela.iterrows():
    linhas.append(f"| {r['Modelo']} | {r['Dados']} | {int(r['Dimensões'])} | {r['Treino (s)']:.2f} | {r['Acurácia']:.2%} | {r['Total para treino (s)']:.2f} |")
linhas += ["", "O total para treino inclui o ajuste do PCA e a transformação do treino nas alternativas com redução; exclui carregamento e avaliação no teste. A floresta usou 100 árvores; o SGD usou `loss=\"hinge\"`, `alpha=0.0001`, `max_iter=1000` e `tol=1e-3`. Todos os modelos usaram `random_state=42`.", ""]
for nome in ("Floresta aleatória", "SGDClassifier"):
    grupo = tabela[tabela["Modelo"] == nome].set_index("Dados")
    original, reduzido = grupo.loc["Originais"], grupo.loc["PCA 95%"]
    fator = original["Treino (s)"] / reduzido["Treino (s)"]
    fator_total = original["Total para treino (s)"] / reduzido["Total para treino (s)"]
    delta = 100 * (reduzido["Acurácia"] - original["Acurácia"])
    comportamento = "mais rápido" if fator >= 1 else "mais lento"
    multiplicador = fator if fator >= 1 else 1 / fator
    linhas += [f"- **{nome}:** o treinamento do classificador com PCA foi **{multiplicador:.2f}× {comportamento}**; a acurácia mudou **{delta:+.2f} p.p.** O fator de aceleração incluindo o custo do PCA foi **{fator_total:.2f}×** (acima de 1 indica ganho)."]
linhas += ["", f"O SGD realizou {sgd_original.n_iter_} épocas nos dados originais e {sgd_pca.n_iter_} nos reduzidos.", "", "**Conclusão:** reduzir dimensões não garante ganho de tempo nem de acurácia. As árvores podem ser prejudicadas pela mudança de representação; um modelo linear pode se beneficiar do menor número de atributos e de mudanças na convergência. Os resultados são de uma única execução, com parâmetros fixos, e os tempos variam conforme o computador e sua carga."]
resumo_markdown = "\n".join(linhas)
display(Markdown(resumo_markdown))

Execução em 2026-10-19T16:52:48-03:00, com Python 3.11.14, NumPy 2.1.3 e scikit-learn 1.5.2; paralelismo limitado a 4.

Foram usadas as primeiras 60.000 imagens do MNIST para treino e as 10.000 restantes para teste. O PCA foi ajustado apenas no treino, com `svd_solver="full"` e `whiten=False`. Os pixels foram convertidos para `float32`, sem padronização adicional.

O PCA reduziu **784 para 154 dimensões**, preservando **95.0196%** da variância do treino. O ajuste levou **1.13 s**, a transformação do treino **0.11 s** e a do teste **0.01 s**.

| Modelo | Dados | Dimensões | Treino (s) | Acurácia no teste | Total para treino (s) |
| --- | --- | ---: | ---: | ---: | ---: |
| Floresta aleatória | Originais | 784 | 6.38 | 97.05% | 6.38 |
| Floresta aleatória | PCA 95% | 154 | 19.11 | 94.88% | 20.34 |
| SGDClassifier | Originais | 784 | 64.97 | 88.47% | 64.97 |
| SGDClassifier | PCA 95% | 154 | 13.95 | 88.13% | 15.19 |

O total para treino inclui o ajuste do PCA e a transformação do treino nas alternativas com redução; exclui carregamento e avaliação no teste. A floresta usou 100 árvores; o SGD usou `loss="hinge"`, `alpha=0.0001`, `max_iter=1000` e `tol=1e-3`. Todos os modelos usaram `random_state=42`.

- **Floresta aleatória:** o treinamento do classificador com PCA foi **3.00× mais lento**; a acurácia mudou **-2.17 p.p.** O fator de aceleração incluindo o custo do PCA foi **0.31×** (acima de 1 indica ganho).
- **SGDClassifier:** o treinamento do classificador com PCA foi **4.66× mais rápido**; a acurácia mudou **-0.34 p.p.** O fator de aceleração incluindo o custo do PCA foi **4.28×** (acima de 1 indica ganho).

O SGD realizou 315 épocas nos dados originais e 175 nos reduzidos.

**Conclusão:** reduzir dimensões não garante ganho de tempo nem de acurácia. As árvores podem ser prejudicadas pela mudança de representação; um modelo linear pode se beneficiar do menor número de atributos e de mudanças na convergência. Os resultados são de uma única execução, com parâmetros fixos, e os tempos variam conforme o computador e sua carga.